# EU LexRef — notebook walkthrough

This notebook runs the same assistant as the web app (`app.py`), step by step:

1. **Set-up**: load the acts in the library (EECC, GDPR, BEREC Regulation).
2. **Ask questions**: answers stream in as they are written, with verified quotations.
3. **Languages**: ask in German and get the answer and the law in German.
4. **Follow-ups**: the assistant keeps the conversation in mind.
5. **Another act**: a GDPR question is routed to the GDPR, then back to the Code.
6. **Outside the library**: the subject is recorded for the author.
7. **Under the hood**: search, act ranking and quotation checks.
8. **Logs and tests**: usage statistics, recorded topics, the automated tests.

The code itself stays in the `eu_law_nli` package; the notebook imports it, so the
notebook, the web app and the terminal chat always behave the same.

**Before you start**
- Open this notebook from the project folder (`EU Lex NLM`), in VS Code, JupyterLab or Jupyter Notebook.
- The API key is read from the `.env` file in that folder. If there is none, the first cell asks for it (it is not shown or saved).
- Each question costs a few cents of API usage.

## 1 · Set-up

In [ ]:
# One-time installation of the packages (uncomment and run once):
# %pip install -r requirements.txt

import os
import sys
from pathlib import Path

ROOT = Path.cwd()  # the project folder, where this notebook is
sys.path.insert(0, str(ROOT))
os.environ.setdefault("NLI_LIVE_CHECK", "off")  # no background EUR-Lex checks here

from eu_law_nli import config, feedback, usage  # reads .env

if not os.environ.get("ANTHROPIC_API_KEY"):
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
print("Project folder:", ROOT)
print("Model:", config.MODEL, "via", config.PROVIDER)

In [ ]:
from eu_law_nli.engine import Engine, Session, continue_in
from eu_law_nli.library import Library
from eu_law_nli.llm import AnthropicLLM

llm = AnthropicLLM()
library = Library()  # one search index across all acts
engines = {doc_id: Engine(llm, doc_id, library=library) for doc_id in library.documents}

for doc_id, e in engines.items():
    print(f"{doc_id:7} {e.doc.short_name} ({e.doc.citation}), {e.version.kind} text of "
          f"{e.version.date}, {len(e.corpora)} languages")

### A small helper to chat

`ask()` sends a question to the selected act and shows the answer as it is written,
then the full reply: summary, explanation, verified quotations and the source line.
`switch()` moves the conversation to another act, keeping the language and the recent
exchanges, as the web app does.

In [ ]:
from IPython.display import Markdown, display

HOME = config.DEFAULT_DOCUMENT  # the Code
state = {"act": HOME, "session": Session(interface="notebook")}


def ask(question: str):
    engine = engines[state["act"]]
    handle = display(Markdown("_Reading the text…_"), display_id=True)

    def progress(markdown: str) -> None:
        handle.update(Markdown(markdown + " ▌"))

    reply = engine.respond(question, state["session"], on_progress=progress)
    handle.update(Markdown(f"**You:** {question}\n\n**{engine.doc.short_name}:**\n\n{reply.text}"))
    return reply


def switch(act: str) -> None:
    state["session"] = continue_in(state["session"], engines[act].doc.short_name, "notebook")
    state["act"] = act
    print("Now answering from:", engines[act].doc.short_name)


def new_conversation() -> None:
    state.update(act=HOME, session=Session(interface="notebook"))
    print("New conversation in", engines[HOME].doc.short_name)

## 2 · Ask a question

In [ ]:
reply = ask("Can my provider charge me for keeping my number when I switch?")

The reply is also available in parts, the way the web app lays it out:

In [ ]:
print("Summary:  ", reply.summary)
print("Articles: ", sorted({q.reference for q in reply.quotes}))
print("Follow-up:", reply.followup or "(none)")
print("Kind:     ", reply.kind)

## 3 · Follow-up question

The assistant reads the recent conversation, so a short follow-up keeps its meaning.

In [ ]:
reply = ask("And how long do I keep that right after my contract ends?")

## 4 · Another language

Write in German: the answer, the quotations (from the official German text) and the
links follow the language.

In [ ]:
reply = ask("Welche Informationen muss ein Anbieter vor Vertragsabschluss bereitstellen?")

## 5 · A question for another act

A question about GDPR fines, asked while the Code is selected: the assistant points to
the GDPR. We switch, ask there, then return to the Code, as the web app does.

In [ ]:
question = "What factors decide the amount of a GDPR fine?"
reply = ask(question)
if reply.suggestion:
    print("Suggested act:", reply.suggestion["name"], "-", reply.suggestion["provisions"])
    switch(reply.suggestion["document"])
    ask(question)
    switch(HOME)  # back to the Code for the next question

## 6 · A subject outside the library

Tax is not covered by any act in the library. The assistant says so, and the subject
is recorded for the author in the topics log (kept 90 days).

In [ ]:
reply = ask("What VAT rate applies to mobile phone contracts in Germany?")
print("Recorded topics:", [t.get("topic_en") for t in reply.outside_topics])

## 7 · Under the hood

**Search.** The keyword search (BM25) over the Code's English text, for one query:

In [ ]:
code_engine = engines[HOME]
seen = set()
for hit in code_engine.retriever.search("number portability charges end-users", k=20):
    if hit.chunk.provision_id in seen:
        continue  # a long provision is split into several searchable chunks
    seen.add(hit.chunk.provision_id)
    provision = code_engine.reference.get(hit.chunk.provision_id)
    print(f"{hit.score:6.1f}  {provision.label:12} {provision.title[:60]}")
    if len(seen) == 5:
        break

**Act ranking.** The same kind of search across every act in the library, used to
suggest a better act:

In [ ]:
for match in library.rank(["administrative fines supervisory authority"])[:3]:
    print(f"{match.document:7} score {match.score:6.1f}   {', '.join(match.provisions[:3])}")

**Quotation check.** Every quotation the model proposes is compared word for word with
the stored official text; those that do not match are dropped before anyone sees them.

In [ ]:
first = ask("What must a regulator do before imposing SMP obligations?")
print(f"Kept {len(first.quotes)} quotation(s), dropped {len(first.dropped_quotes)}.")
for q in first.quotes:
    print(f"- {q.reference}: {q.text[:90]}…")

## 8 · Logs and tests

In [ ]:
s = usage.summary(usage.read_all())
print(f"Messages: {s['turns']} in {s['conversations']} conversations")
print("Languages:", dict(s["languages"].most_common(5)))
print("Articles quoted most:", dict(s["articles"].most_common(5)))
print(f"Estimated API cost: ${s['cost_usd']:.2f}")
print()
print("Latest topics recorded for the author:")
for entry in feedback.read_all()[-3:]:
    print("-", entry["question"], "->", [feedback.describe(t) for t in entry["topics"]])

In [ ]:
# The automated tests (no API calls, about 5 seconds):
!python -m unittest -q tests.test_prototype

## The web app

The same assistant as a web page: run this in a terminal in the project folder, not in
the notebook (it keeps running until you stop it):

```
streamlit run app.py
```